### CELL 1: Environment Setup & Paths

In [25]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# os.environ["CACHE_LOOKBACK"] = "10"
os.environ["CACHE_LOOKBACK"] = "20"
os.environ["CACHE_START_DATE"] = "2026-01-01"

# ============================================================================
# --- ENVIRONMENT PATH CONFIGURATION ---
# Adjust paths if your directory structure differs.
# ============================================================================
project_path = Path("C:/Users/ping/Files_win10/python/py311/stocks")
codebase_path = project_path / "notebooks_RLVR_v2"
data_path = project_path / "data" / "processed"
output_path = codebase_path / "output"

# Add codebase to system path so imports resolve cleanly
if str(codebase_path) not in sys.path:
    sys.path.append(str(codebase_path))

# Load standard configs
# CacheConfig automatically reads CACHE_LOOKBACK and CACHE_START_DATE from os.environ
from core.settings import CacheConfig, TradingConfig

print("--- Dynamic Config Verification ---")
print(f"Lookback detected  : {CacheConfig.LOOKBACK}")
print(f"Start Date detected: {CacheConfig.START_DATE}")

--- Dynamic Config Verification ---
Lookback detected  : 20
Start Date detected: 2026-01-01


### CELL 2: Load Datasets

In [26]:
print("Loading raw data. Please wait...")

# Load raw datasets
df_ohlcv = pd.read_parquet(project_path / "data" / "df_OHLCV_stocks_etfs.parquet")
macro_df = pd.read_parquet(data_path / "macro_df.parquet")
features_df = pd.read_parquet(data_path / "features_df.parquet")

df_close = df_ohlcv["Adj Close"].unstack(level=0)
trading_calendar = pd.DatetimeIndex(df_close.dropna(how="all").index.sort_values())

# Resolve the filename dynamically via CacheConfig
cache_file_name = CacheConfig.get_filename()
cache_file_path = output_path / cache_file_name

if cache_file_path.exists():
    feature_cube = pd.read_parquet(cache_file_path)
    print(f"Loaded feature cube: {cache_file_name} (Shape: {feature_cube.shape})")
else:
    print(f"[ERROR] Cache file not found at: {cache_file_path}")
    feature_cube = None

Loading raw data. Please wait...
Loaded feature cube: alpha_cache_20d_2026.parquet (Shape: (99582, 11))


In [27]:
feature_cube

20d_Log Price Gain  20d_Sharpe (TRP)  20d_Momentum (21d)  \
Date       Ticker                                                             
2026-01-02 A                -0.078805         -0.215628           -0.072980   
           AA                0.248488          0.354379            0.364054   
           AAL               0.061271          0.104846            0.087079   
           AAPL             -0.047348         -0.159386           -0.053042   
           ABBV             -0.004048         -0.009772            0.022016   
...                               ...               ...                 ...   
2026-06-04 ZBH               0.045548          0.085065            0.044332   
           ZBRA              0.066180          0.084435            0.074828   
           ZM                0.000761          0.015140           -0.035655   
           ZS               -0.026051          0.037372           -0.043152   
           ZTS              -0.335502         -0.278090           -0.293407   

                   20d_Info Ratio (63d)  20d_Oversold (-RSI)  \
Date       Ticker                                              
2026-01-02 A                  -0.031678           -41.393906   
           AA                  0.241136           -77.239978   
           AAL                 0.187154           -58.414461   
           AAPL                0.055573           -43.834556   
           ABBV               -0.043662           -53.954973   
...                                 ...                  ...   
2026-06-04 ZBH                -0.130362           -55.762076   
           ZBRA               -0.014438           -52.358705   
           ZM                  0.117286           -55.614349   
           ZS                 -0.047509           -43.714098   
           ZTS                -0.241823           -33.587569   

                   20d_Dip Buyer (-dd_21)  20d_Range Position (20d)  \
Date       Ticker                                                     
2026-01-02 A                     0.075779                  0.184438   
           AA                   -0.000000                  0.994951   
           AAL                   0.047970                  0.591549   
           AAPL                  0.046244                  0.228320   
           ABBV                  0.006627                  0.770153   
...                                   ...                       ...   
2026-06-04 ZBH                  -0.000000                  0.819264   
           ZBRA                  0.056064                  0.617460   
           ZM                    0.059617                  0.508256   
           ZS                    0.267281                  0.187845   
           ZTS                   0.285021                  0.370717   

                   20d_Return Autocorr (15d)  20d_Low Volatility (-ATRP)  \
Date       Ticker                                                          
2026-01-02 A                       -0.169489                   -0.024528   
           AA                       0.004069                   -0.036758   
           AAL                     -0.228331                   -0.033850   
           AAPL                    -0.344589                   -0.017517   
           ABBV                    -0.424249                   -0.020958   
...                                      ...                         ...   
2026-06-04 ZBH                      0.480329                   -0.029968   
           ZBRA                     0.115117                   -0.041324   
           ZM                      -0.261631                   -0.043538   
           ZS                      -0.010319                   -0.063483   
           ZTS                     -0.049789                   -0.052144   

                   20d_OBV Divergence (5d)  20d_Convexity  
Date       Ticker                                          
2026-01-02 A                     -1.019142      -0.000640  
           AA                    -1.017137       0.004127  
     

### CELL 3:  Input Selection & Dynamic Universe Screening

In [28]:
# ============================================================================
# --- VERIFICATION INPUTS ---
# Define the target Date and Ticker you wish to verify.
# ============================================================================
# target_date_str = "2026-01-02"  # Must be on or after CacheConfig.START_DATE
# target_ticker = "A"

target_date_str = "2026-06-04"  # Must be on or after CacheConfig.START_DATE
target_ticker = "ZS"


target_date = pd.Timestamp(target_date_str)

# Resolve lookback and prefix dynamically
lookback_lb = CacheConfig.LOOKBACK
prefix = f"{lookback_lb}d_"

print(
    f"Verifying features for Ticker: {target_ticker} on Date: {target_date.date()} using Lookback: {prefix}\n"
)

# 1. Replicate filter_universe() to identify active candidates on this date
day_features = features_df.xs(target_date, level="Date")
vol_cutoff = 1_000_000  # min_median_dollar_volume

# min_liquidity_percentile = 0.40
vol_cutoff = max(vol_cutoff, day_features["RollMedDollarVol"].quantile(0.40))

mask = (
    (day_features["RollMedDollarVol"] >= vol_cutoff)
    & (day_features["RollingStalePct"] <= 0.05)  # max_stale_pct
    & (day_features["RollingSameVolCount"] <= 10)  # max_same_vol_count
)

candidates = day_features[mask].index.tolist()
print(f"Total screened candidates on {target_date.date()}: {len(candidates)}")
print(
    f"Is target ticker '{target_ticker}' in the active universe? {target_ticker in candidates}"
)

# 2. Determine Timeline Dates dynamically using lookback_lb
decision_idx = trading_calendar.searchsorted(target_date)
start_idx = int(decision_idx - lookback_lb)
start_date = trading_calendar[start_idx]

# Extract exact active windows
full_window_dates = trading_calendar[
    (trading_calendar >= start_date) & (trading_calendar <= target_date)
]
active_dates = full_window_dates[1:]

print(f"\nLookback ({prefix[:-1]}):")
print(f"  Start Date    : {start_date.date()} (index: {start_idx})")
print(f"  Decision Date : {target_date.date()} (index: {decision_idx})")
print(f"  Active Dates Count: {len(active_dates)} trading days")

Verifying features for Ticker: ZS on Date: 2026-06-04 using Lookback: 20d_

Total screened candidates on 2026-06-04: 943
Is target ticker 'ZS' in the active universe? True

Lookback (20d):
  Start Date    : 2026-05-06 (index: 16193)
  Decision Date : 2026-06-04 (index: 16213)
  Active Dates Count: 20 trading days


### CELL 4: Step-by-Step Dynamic Independent Calculations

In [29]:
from typing import Any, Union, cast
import pandas as pd
import numpy as np

# Create a dictionary to hold our independent calculations
independent_vals = {}


def _to_float(val: Any) -> float:
    """Safely extracts a single float value from a pandas indexing result."""
    if isinstance(val, pd.Series):
        # Cast the broad Scalar union to Any to satisfy the float() type check
        return float(cast(Any, val.iloc[0]))
    if isinstance(val, pd.DataFrame):
        # Cast the broad Scalar union to Any to satisfy the float() type check
        return float(cast(Any, val.iloc[0, 0]))
    return float(val)


# Helper to retrieve feature values safely regardless of index orientation
def get_feat_val(ticker: str, date: pd.Timestamp, col: str) -> float:
    try:
        names = features_df.index.names
        if "Ticker" in names and "Date" in names:
            t_idx = names.index("Ticker")
            d_idx = names.index("Date")

            # 1. Annotate key to avoid list[None] type inference errors
            key: list[Any] = [None, None]
            key[t_idx] = ticker
            key[d_idx] = date

            # 2. Cast tuple(key) to avoid any MultiIndex indexing complaints
            val = features_df.loc[cast(tuple[Any, ...], tuple(key)), col]
            return _to_float(val)
        else:
            # 3. Handle the union return type of .xs() safely via type-narrowing
            xs_result = features_df.xs(date, level="Date")
            if isinstance(xs_result, pd.DataFrame):
                return _to_float(xs_result.loc[ticker, col])
            else:
                # Fallback if xs_result is a Series
                return float(xs_result.loc[ticker])

    except Exception as e:
        print(f"[WARNING] Failed to fetch {col} for {ticker} on {date.date()}: {e}")
        return float("nan")


# ----------------------------------------------------------------------------
# 1. Log Price Gain
# ----------------------------------------------------------------------------
ticker_prices = df_close.loc[full_window_dates, target_ticker]
clean_prices = ticker_prices.dropna()
if len(clean_prices) >= 2 and clean_prices.iloc[0] > 0 and clean_prices.iloc[-1] > 0:
    independent_vals[f"{prefix}Log Price Gain"] = float(
        np.log(clean_prices.iloc[-1] / clean_prices.iloc[0])
    )
else:
    independent_vals[f"{prefix}Log Price Gain"] = 0.0

# ----------------------------------------------------------------------------
# 2. Sharpe (TRP)
# ----------------------------------------------------------------------------
if features_df.index.names == ["Ticker", "Date"]:
    feat_window = features_df.loc[pd.IndexSlice[candidates, active_dates], :]
else:
    feat_window = features_df.loc[pd.IndexSlice[active_dates, candidates], :]

lookback_close_subset = df_close.loc[full_window_dates, candidates]
lookback_returns_subset = lookback_close_subset.pct_change()

obs_trp = feat_window["TRP"].groupby(level="Ticker").mean().reindex(candidates)

ret_arr = lookback_returns_subset.to_numpy(dtype=float)
vol_arr = obs_trp.to_numpy(dtype=float)

avg_ret = np.nanmean(ret_arr, axis=0)
avg_vol = np.maximum(vol_arr, 1e-8)
res_arr = avg_ret / avg_vol

sharpe_trp_series = pd.Series(res_arr, index=candidates)
independent_vals[f"{prefix}Sharpe (TRP)"] = sharpe_trp_series.loc[target_ticker]

# ----------------------------------------------------------------------------
# 3. Momentum (21d)
# ----------------------------------------------------------------------------
independent_vals[f"{prefix}Momentum (21d)"] = get_feat_val(
    target_ticker, target_date, "Mom_21"
)

# ----------------------------------------------------------------------------
# 4. Info Ratio (63d)
# ----------------------------------------------------------------------------
independent_vals[f"{prefix}Info Ratio (63d)"] = get_feat_val(
    target_ticker, target_date, "IR_63"
)

# ----------------------------------------------------------------------------
# 5. Oversold (-RSI)
# ----------------------------------------------------------------------------
independent_vals[f"{prefix}Oversold (-RSI)"] = -get_feat_val(
    target_ticker, target_date, "RSI"
)

# ----------------------------------------------------------------------------
# 6. Dip Buyer (-dd_21)
# ----------------------------------------------------------------------------
independent_vals[f"{prefix}Dip Buyer (-dd_21)"] = -get_feat_val(
    target_ticker, target_date, "DD_21"
)

# ----------------------------------------------------------------------------
# 7. Range Position (20d)
# ----------------------------------------------------------------------------
independent_vals[f"{prefix}Range Position (20d)"] = get_feat_val(
    target_ticker, target_date, "Range_Pos_20"
)

# ----------------------------------------------------------------------------
# 8. Return Autocorr (15d)
# ----------------------------------------------------------------------------
independent_vals[f"{prefix}Return Autocorr (15d)"] = get_feat_val(
    target_ticker, target_date, "AutoCorr_15"
)

# ----------------------------------------------------------------------------
# 9. Low Volatility (-ATRP)
# ----------------------------------------------------------------------------
obs_atrp = feat_window["ATRP"].groupby(level="Ticker").mean().reindex(candidates)
independent_vals[f"{prefix}Low Volatility (-ATRP)"] = -obs_atrp.loc[target_ticker]

# ----------------------------------------------------------------------------
# 10. OBV Divergence (5d)
# ----------------------------------------------------------------------------
feat_now = features_df.xs(target_date, level="Date").reindex(candidates)
slope_v = feat_now["Slope_V_5"]
slope_p = feat_now["Slope_P_5"]


def calculate_zscore(series: pd.Series) -> pd.Series:
    m = float(series.mean())
    s = float(series.std())
    denom = s if (s != 0 and not np.isnan(s)) else 1.0
    return (series - m) / denom


z_v = calculate_zscore(slope_v)
z_p = calculate_zscore(slope_p)
obv_div_series = z_v - z_p
independent_vals[f"{prefix}OBV Divergence (5d)"] = obv_div_series.loc[target_ticker]

# ----------------------------------------------------------------------------
# 11. Convexity
# ----------------------------------------------------------------------------
independent_vals[f"{prefix}Convexity"] = get_feat_val(
    target_ticker, target_date, "Convexity"
)

print("Independent calculations complete.")

Independent calculations complete.


### CELL 5: Dynamic Comparison and Verification Report

In [31]:
# Load pre-calculated values from the feature cube
if feature_cube is not None:
    try:
        cache_row = feature_cube.loc[(target_date, target_ticker)]
    except KeyError:
        print(
            f"[ERROR] Could not find {(target_date.date(), target_ticker)} in feature_cube."
        )
        cache_row = None
else:
    cache_row = None

# Build comparison report
report_data = []
for feature_name, ind_val in independent_vals.items():
    cache_val = cache_row[feature_name] if cache_row is not None else np.nan
    diff = (
        abs(ind_val - cache_val)
        if not (np.isnan(ind_val) or np.isnan(cache_val))
        else 0.0
    )

    # Check if they match within a tight tolerance
    status = (
        "MATCH"
        if (diff < 1e-7 or (np.isnan(ind_val) and np.isnan(cache_val)))
        else "MISMATCH"
    )

    report_data.append(
        {
            "Feature Name": feature_name,
            "Independent Val": ind_val,
            "Feature Cube Val": cache_val,
            "Difference": diff,
            "Status": status,
        }
    )

report_df = pd.DataFrame(report_data)

# Print formatting
print(f"=== VERIFICATION REPORT FOR {target_ticker} ON {target_date.date()} ===")
print(f"Config: {cache_file_name}")
display(
    report_df.style.format(
        {
            "Independent Val": "{:.6f}",
            "Feature Cube Val": "{:.6f}",
            "Difference": "{:.2e}",
        }
    )
)

=== VERIFICATION REPORT FOR ZS ON 2026-06-04 ===
Config: alpha_cache_20d_2026.parquet


,Feature Name,Independent Val,Feature Cube Val,Difference,Status
0,20d_Log Price Gain,-0.026051,-0.026051,0.00e+00,MATCH
1,20d_Sharpe (TRP),0.037372,0.037372,0.00e+00,MATCH
2,20d_Momentum (21d),-0.043152,-0.043152,0.00e+00,MATCH
3,20d_Info Ratio (63d),-0.047509,-0.047509,0.00e+00,MATCH
4,20d_Oversold (-RSI),-43.714098,-43.714098,0.00e+00,MATCH
5,20d_Dip Buyer (-dd_21),0.267281,0.267281,0.00e+00,MATCH
6,20d_Range Position (20d),0.187845,0.187845,0.00e+00,MATCH
7,20d_Return Autocorr (15d),-0.010319,-0.010319,0.00e+00,MATCH
8,20d_Low Volatility (-ATRP),-0.063483,-0.063483,0.00e+00,MATCH
9,20d_OBV Divergence (5d),-0.854432,-0.854432,0.00e+00,MATCH
